# Rossmann data preparation and diagnostics

Purpose: verify the immutable Phase 1 source snapshot, prepare separate joined train/test Parquet tables, and inspect the approved sparse-row and missing-value policies. No future target or fabricated row is created.

Source SHA-256 values are checked against `docs/DATA_ACQUISITION.md` by `rossmann-prepare`. Environment: Python 3.14 with dependencies from `pyproject.toml`. Run from the repository root with `python scripts/prepare_data.py`; generated outputs are `data/interim/train.parquet`, `test.parquet`, and `preparation_manifest.json` (ignored).

In [ ]:
from rossmann_forecasting.data.preparation import run_preparation

manifest = run_preparation()
{name: details["rows"] for name, details in manifest["outputs"].items()}

In [ ]:
import pandas as pd

train = pd.read_parquet("data/interim/train.parquet")
test = pd.read_parquet("data/interim/test.parquet")
{
    "train_rows": len(train),
    "test_rows": len(test),
    "train_unique_store_dates": not train.duplicated(["Store", "Date"]).any(),
    "closed_train_rows_retained": int(train["Open"].eq(0).sum()),
    "open_zero_sales_rows_retained": int((train["Open"].eq(1) & train["Sales"].eq(0)).sum()),
    "test_open_nulls_retained": int(test["Open"].isna().sum()),
    "test_has_target": "Sales" in test.columns,
}